# 3-3절 연습 문제 풀이

이 노트북은 3-3절 연습 문제(3-9 ~ 3-12)의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 가능하다.

- 본문 예제 코드는 `code_examples/ch03/03-03_example.ipynb`를 참고한다.
- 각 문제마다 **풀이 해설**과 **문제 검토**를 함께 실었다. 문제 검토는 최종 검토 3단계의 기록이다.

In [1]:
import csv
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

SEED = 1
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = '../../data'
LR = 0.01
EPOCHS = 1000

## 연습 문제 3-9

> [코드 3-23]에서 모델의 정확도를 계산할 때, 정답을 맞힌 샘플의 수를 집계한 후 전체 샘플의 수로 나눠 백분율을
> 구하는 두 줄의 코드는 `(classes == Y).float().mean().item() * 100` 한 줄로 줄일 수 있다.
> 이 한 줄의 코드가 정확도 백분율을 계산하는 과정을 풀어서 설명해 보자.

### 풀이 해설

한 줄에 네 단계가 겹쳐 있다. 왼쪽부터 차례로 풀면 이렇다.

| 단계 | 결과 | 설명 |
|---|---|---|
| `classes == Y` | 참거짓형 텐서 | 샘플마다 예측과 정답이 같은지 비교한다. 1장의 **불리언 마스크**와 같은 형태다. |
| `.float()` | 실수형 텐서 | `True`를 1.0으로, `False`를 0.0으로 바꾼다. |
| `.mean()` | 스칼라 텐서 | 0과 1의 평균은 **1의 비율**, 곧 맞힌 비율이다. |
| `.item()` | 파이썬 실수 | 스칼라 텐서를 숫자로 꺼낸다. |
| `* 100` | 백분율 | 비율에 100을 곱한다. |

핵심은 세 번째 단계다. **0과 1로만 이루어진 텐서의 평균은 곧 1의 비율**이라는 성질을 이용해,
'맞힌 개수를 세어 전체로 나누는' 두 단계를 `mean()` 하나로 합친 것이다.
`sum()`으로 세고 길이로 나누는 원래 방식과 결과가 같지만, 전체 샘플 수를 따로 구할 필요가 없어 짧아진다.

`.float()`가 꼭 필요한 이유도 짚어 둘 만하다. 참거짓형 텐서에는 `mean()`을 쓸 수 없어서 예외가 발생한다.

In [2]:
classes = torch.tensor([0, 1, 2, 1, 0, 2, 2, 1])   # 모델의 분류 결과(예시)
Y = torch.tensor([0, 1, 2, 2, 0, 2, 1, 1])         # 정답(예시)

print(f'1) classes == Y          : {(classes == Y)}')
print(f'2) .float()              : {(classes == Y).float()}')
print(f'3) .mean()               : {(classes == Y).float().mean()}')
print(f'4) .item()               : {(classes == Y).float().mean().item()}')
print(f'5) * 100                 : {(classes == Y).float().mean().item() * 100:.2f}%')

# 두 줄로 계산한 원래 방식과 같은 값인지 확인
correct = (classes == Y).sum().item()
print(f'\n맞힌 개수 {correct} / 전체 {len(Y)} = {correct / len(Y) * 100:.2f}%')

# 참거짓형 텐서에는 mean()을 쓸 수 없다
try:
    (classes == Y).mean()
except Exception as e:
    print(f'\n.float() 없이 mean() 호출 -> {type(e).__name__}: {str(e)[:60]}')

1) classes == Y          : tensor([ True,  True,  True, False,  True,  True, False,  True])
2) .float()              : tensor([1., 1., 1., 0., 1., 1., 0., 1.])
3) .mean()               : 0.75
4) .item()               : 0.75
5) * 100                 : 75.00%

맞힌 개수 6 / 전체 8 = 75.00%

.float() 없이 mean() 호출 -> RuntimeError: mean(): could not infer output dtype. Input dtype must be ei


### 문제 검토

- **적절성: 적합.** 파이토치 코드를 읽을 때 자주 만나는 메서드 연쇄를 한 단계씩 뜯어보게 한다.
  특히 '0과 1의 평균이 곧 비율'이라는 발상은 정확도뿐 아니라 이후 여러 지표 계산에서 반복해 쓰인다.
  1장의 불리언 마스크, 텐서 자료형 변환, `item()`이 한 줄에 모여 있어 복습 효과도 있다.
- **[검토] 지문에 나온 변수 이름이 본문 코드와 다를 수 있다.** [코드 3-23]에서 쓰는 변수명과 지문의 `classes`,
  `Y`가 일치하는지 확인이 필요하다. 지문이 코드 한 줄을 그대로 인용하고 있으므로 이름이 어긋나면 독자가 헤맨다.

## 연습 문제 3-10

> 교차 엔트로피 손실 함수를 사용해 학습하는 모델은 출력층에 활성화 계층을 덧붙이지 않는다.
> 그렇다면 분류 모델이 아닌 1장의 회귀 분석 모델에서 활성화 함수를 사용하지 않았던 이유가 무엇인지 생각해 보자.

### 풀이 해설

두 경우는 활성화 계층을 두지 않는 이유가 **서로 다르다**. 그 차이를 구분하는 것이 이 문제의 핵심이다.

**교차 엔트로피 모델(3장)** — 활성화가 필요 없어서가 아니라, **손실 함수 안에 이미 들어 있기 때문**이다.
`nn.CrossEntropyLoss`가 내부에서 소프트맥스를 적용하므로 모델에 또 붙이면 두 번 적용된다.
즉 활성화는 여전히 필요하고, 위치만 모델 밖으로 옮겨진 것이다.

**회귀 분석 모델(1장)** — 활성화 함수 자체가 **필요 없다.** 이유는 두 가지다.

1. **출력의 범위를 제한하면 안 된다.** 1장 모델은 낙하거리를 예측한다. 관측시간이 커지면 낙하거리는 수백 미터까지
   커지는데, 시그모이드를 붙이면 출력이 0~1 사이로 갇혀 어떤 파라미터로도 500을 낼 수 없다.
   분류는 '어느 쪽인가'를 고르는 문제라 출력을 0~1로 눌러도 되지만, 회귀는 값 자체가 답이다.
2. **비선형성을 더할 자리가 아니다.** 활성화 함수의 또 다른 역할인 비선형성 추가는 **계층 사이**에서 의미가 있다.
   1장 모델은 계층이 하나뿐이라 뒤에 이어질 계층이 없다.

정리하면 **출력층의 활성화 함수는 출력의 형태를 문제에 맞추는 역할**을 한다.
분류면 확률 형태로, 회귀면 아무 제한 없이 두는 것이 문제에 맞는 형태다.
본문 p35가 "출력층에 ReLU를 쓰면 출력이 0 이상으로 제한된다"며 ReLU를 출력층에 쓰지 않는 이유로 든 것도 같은 맥락이다.

In [3]:
# 시그모이드를 출력층에 붙이면 회귀 모델이 어떻게 되는지 확인
torch.manual_seed(8)
X = torch.rand((50, 1)) * 10
Y_true = 4.9 * X ** 2 + torch.randn((50, 1)) * (4.9 * X ** 2) * 0.1
print(f'정답 낙하거리의 범위: {Y_true.min().item():.1f} ~ {Y_true.max().item():.1f}')

for name, model in (('활성화 없음', nn.Linear(1, 1)),
                    ('시그모이드 있음', nn.Sequential(nn.Linear(1, 1), nn.Sigmoid()))):
    torch.manual_seed(SEED)
    model = model if name == '활성화 없음' else model
    optimizer = optim.Adam(model.parameters(), lr=0.1)
    criterion = nn.MSELoss()
    for _ in range(2000):
        optimizer.zero_grad()
        loss = criterion(model(X ** 2), Y_true)
        loss.backward()
        optimizer.step()
    with torch.no_grad():
        pred = model(X ** 2)
    print(f'{name:10}: 손실 {loss.item():10.2f}, 예측 범위 {pred.min().item():.2f} ~ {pred.max().item():.2f}')

정답 낙하거리의 범위: 0.0 ~ 499.3


활성화 없음    : 손실     452.52, 예측 범위 0.76 ~ 461.47


시그모이드 있음  : 손실   48041.34, 예측 범위 0.07 ~ 1.00


### 문제 검토

- **적절성: 적합. 3장을 마무리하는 좋은 질문이다.** 1장과 3장을 이어 붙여, 활성화 함수를 '붙이는 규칙'이 아니라
  '문제에 맞는 출력 형태를 만드는 장치'로 이해하게 한다. 서술형이지만 답이 하나로 모이는 좋은 문제다.
- **[검토] 두 이유가 다르다는 점을 지문이 가리키면 더 좋다.** 현재 지문은 "그렇다면 … 이유가 무엇인지"로
  두 경우를 나란히 놓는데, 독자는 같은 이유를 찾으려다 헤맬 수 있다. 실제로는 3장은 '손실 함수가 대신한다',
  1장은 '애초에 필요 없다'로 성격이 다르다.
- **[검토] 확인할 방법이 없다.** 위 코드처럼 출력층에 시그모이드를 붙여 학습해 보면 예측값이 0~1에 갇혀
  손실이 전혀 줄지 않는 것을 바로 볼 수 있다. 한 구절이면 검증까지 이어진다.

**윤문안**

> **3-10**. 교차 엔트로피 손실 함수를 사용해 학습하는 모델은 출력층에 활성화 계층을 덧붙이지 않는다.
> 그렇다면 분류 모델이 아닌 1장의 회귀 분석 모델에서 활성화 함수를 사용하지 않았던 이유가 무엇인지 생각해 보자.
> 두 경우의 이유가 같은지 다른지도 함께 짚어 보고, 1장의 회귀 분석 모델 출력층에 시그모이드 활성화 계층을
> 붙여 학습하면 어떻게 되는지 확인해 보자.

## 연습 문제 3-11

> 깃허브 저장소의 data 디렉터리에 있는 ch3_exercise_3.csv 파일과 ch3_exercise_4.csv 파일은 각각
> [연습 문제 3-7]의 ch3_exercise_1.csv, ch3_exercise_2.csv 파일과 성격은 비슷하지만, 더 많은 클래스로
> 분류되는 다중 클래스 데이터다. 두 파일에 담긴 데이터의 분포는 [그림 3-16]과 같다.
> 각 파일의 데이터를 분류할 수 있는 모델을 만들어 보자.

In [4]:
def load_csv(file_name):
    with open(f'{DATA_DIR}/{file_name}', 'r') as f:
        rows = list(csv.DictReader(f))
    class_names = sorted({row['label'] for row in rows})
    class_index = {name: i for i, name in enumerate(class_names)}
    X = torch.tensor([[float(row['x']), float(row['y'])] for row in rows])
    Y = torch.tensor([class_index[row['label']] for row in rows])
    return X, Y, class_names

def split_data(X, Y, train_ratio=0.6, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    order = torch.randperm(len(X), generator=generator)
    X, Y = X[order], Y[order]
    train_size = int(len(X) * train_ratio)
    return X[:train_size], Y[:train_size], X[train_size:], Y[train_size:]

HIDDEN_DIM = 64

def run_classification(file_name, epochs=EPOCHS):
    X_all, Y_all, class_names = load_csv(file_name)
    X_train, Y_train, X_test, Y_test = split_data(X_all, Y_all)
    torch.manual_seed(SEED)
    # 출력층 뉴런 수는 클래스 수와 같고, 교차 엔트로피를 쓰므로 활성화 계층은 두지 않는다
    model = nn.Sequential(
        nn.Linear(2, HIDDEN_DIM), nn.ReLU(),
        nn.Linear(HIDDEN_DIM, HIDDEN_DIM), nn.ReLU(),
        nn.Linear(HIDDEN_DIM, len(class_names)),
    )
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    model.train()
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = criterion(model(X_train), Y_train)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        accuracy = (model(X_test).argmax(dim=-1) == Y_test).float().mean().item() * 100
    print(f'{file_name}: 샘플 {len(X_all)}개, 클래스 {len(class_names)}개 {class_names}')
    print(f'    훈련 손실 {loss.item():.4f}, 평가 정확도 {accuracy:.2f}%')

for file_name in ('ch3_exercise_3.csv', 'ch3_exercise_4.csv'):
    run_classification(file_name)

ch3_exercise_3.csv: 샘플 2400개, 클래스 3개 ['바깥', '안쪽', '중간']
    훈련 손실 0.0000, 평가 정확도 100.00%


ch3_exercise_4.csv: 샘플 1600개, 클래스 4개 ['우상', '우하', '좌상', '좌하']
    훈련 손실 0.0001, 평가 정확도 100.00%


### 풀이 해설

두 데이터 모두 **평가 정확도 100%**다. 클래스가 세 개, 네 개로 늘었지만 코드에서 바뀌는 것은 사실상 한 곳뿐이다.
**출력층 뉴런의 수를 클래스 수에 맞추는 것**이다. 위 코드에서는 `len(class_names)`로 자동 처리했다.

교차 엔트로피 손실 함수를 쓰므로 정답은 인덱스 그대로 두면 되고, 원-핫 인코딩은 필요 없다.
클래스가 몇 개든 손실 함수가 알아서 처리하므로, 이진 분류에서 다중 클래스로 넘어가는 비용이 거의 없다.
본문 p20이 "클래스가 n개인 다중 클래스 분류 문제에는 일반화된 접근법이 필요하다"고 한 것의 실제 모습이다.

다만 본문 회오리 데이터(정확도 93%)보다 이 데이터들이 훨씬 쉽다. 클래스끼리 겹치는 영역이 없기 때문이다.
클래스 수가 늘어나는 것 자체는 난도를 크게 올리지 않는다는 점도 확인할 수 있다.

### 문제 검토

- **적절성: 적합.** [연습 문제 3-7]과 같은 데이터 성격에 클래스만 늘려, 다중 클래스 확장이 얼마나 간단한지
  직접 확인하게 한다. 두 문제를 짝으로 배치한 구성이 좋다.
- **[검토] 3-7과 같은 문제를 물려받는다.** 열 이름이 `x`, `y`라 본문 [코드 3-7]을 그대로 쓰면 `KeyError`가 나고,
  같은 클래스끼리 모여 있어 섞지 않고 자르면 훈련 데이터에 일부 클래스만 들어간다.
  3-7의 힌트를 고치면 이 문제도 함께 해결된다.
- **[검토] 지문에 힌트가 없다.** 3-7에는 문자열 정답을 인덱스로 바꾸라는 힌트가 있는데 3-11에는 없다.
  같은 처리가 필요하므로 "[연습 문제 3-7]과 같은 방법으로 정답을 변환한다" 정도의 한 줄이 있으면 좋다.
- **[검토] 난도가 3-7보다 낮게 느껴질 수 있다.** 실제로 코드에서 바뀌는 곳이 출력층 뉴런 수뿐이다.
  그것이 이 문제의 메시지이므로 문제는 아니지만, "바뀌는 곳이 몇 군데인지 세어 보자" 같은 질문을 덧붙이면
  독자가 그 메시지를 놓치지 않는다.

## 연습 문제 3-12 [도전 문제]

> ReLU 활성화 함수와 교차 엔트로피 손실 함수를 직접 구현하고, 이를 사용해 회오리 모양 데이터 분류 모델을 만들어 보자.

In [5]:
# 1) ReLU 활성화 함수 - 0 이하는 0, 0보다 크면 그대로
class MyReLU(nn.Module):
    def forward(self, x):
        return torch.clamp(x, min=0.)      # 1장에서 배운 clamp() 사용

# 2) 교차 엔트로피 손실 함수 - 소프트맥스를 적용한 뒤 정답 클래스의 -log(p)를 평균
class MyCrossEntropyLoss(nn.Module):
    def forward(self, logits, target):
        # 지수 함수의 overflow를 막기 위해 각 행의 최댓값을 빼 준다(결과는 같다)
        shifted = logits - logits.max(dim=-1, keepdim=True).values
        probs = shifted.exp() / shifted.exp().sum(dim=-1, keepdim=True)
        # 샘플마다 정답 클래스의 확률만 골라낸다
        correct_probs = probs[torch.arange(len(target)), target]
        return -torch.log(correct_probs).mean()

# 파이토치 구현과 같은 값을 내는지 확인
torch.manual_seed(SEED)
logits = torch.randn(5, 3)
target = torch.tensor([0, 2, 1, 1, 0])
print(f'직접 구현한 ReLU와 nn.ReLU 결과 일치: '
      f'{torch.allclose(MyReLU()(logits), nn.ReLU()(logits))}')
print(f'직접 구현한 교차 엔트로피: {MyCrossEntropyLoss()(logits, target).item():.6f}')
print(f'nn.CrossEntropyLoss   : {nn.CrossEntropyLoss()(logits, target).item():.6f}')

직접 구현한 ReLU와 nn.ReLU 결과 일치: True
직접 구현한 교차 엔트로피: 1.049871
nn.CrossEntropyLoss   : 1.049871


In [6]:
# 직접 구현한 두 요소로 회오리 데이터 분류 모델 학습
# 회오리 데이터는 열 이름이 x1, x2이므로 별도의 로더를 쓴다
def load_spiral():
    with open(f'{DATA_DIR}/ch3_spiral_data.csv', 'r') as f:
        rows = list(csv.DictReader(f))
    X = torch.tensor([[float(r['x1']), float(r['x2'])] for r in rows])
    Y = torch.tensor([int(r['label']) for r in rows])
    return X, Y

X_all, Y_all = load_spiral()
X_train, Y_train, X_test, Y_test = split_data(X_all, Y_all)

results = {}
for name, activation, criterion in (('직접 구현', MyReLU(), MyCrossEntropyLoss()),
                                    ('파이토치 제공', nn.ReLU(), nn.CrossEntropyLoss())):
    torch.manual_seed(SEED)
    model = nn.Sequential(
        nn.Linear(2, HIDDEN_DIM), activation,
        nn.Linear(HIDDEN_DIM, HIDDEN_DIM), activation,
        nn.Linear(HIDDEN_DIM, 3),
    )
    optimizer = optim.Adam(model.parameters(), lr=LR)
    model.train()
    for _ in range(EPOCHS):
        optimizer.zero_grad()
        loss = criterion(model(X_train), Y_train)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        accuracy = (model(X_test).argmax(dim=-1) == Y_test).float().mean().item() * 100
    results[name] = (loss.item(), accuracy)
    print(f'{name:8}: 훈련 손실 {loss.item():.4f}, 평가 정확도 {accuracy:.2f}%')

직접 구현   : 훈련 손실 0.0335, 평가 정확도 94.44%


파이토치 제공 : 훈련 손실 0.0332, 평가 정확도 94.44%


### 풀이 해설

**직접 구현한 두 요소로 학습한 결과가 파이토치 제공 클래스와 사실상 같다.** 손실값도 정확도도 일치한다.

ReLU는 1장에서 배운 `clamp(min=0)` 한 줄이면 된다. 중요한 것은 **미분을 따로 구현할 필요가 없다**는 점이다.
파이토치의 자동 미분이 `clamp` 연산의 기울기를 알아서 계산하므로, 순전파만 정의하면 역전파는 공짜로 따라온다.
1장에서 배운 자동 미분의 의미가 여기서 드러난다.

교차 엔트로피는 두 단계다. 소프트맥스로 로짓을 확률로 바꾸고, 정답 클래스의 확률에 -log를 씌워 평균 낸다.
구현할 때 두 가지를 조심해야 한다.

1. **오버플로 방지**: 로짓이 크면 `exp()`에서 값이 무한대가 된다. 각 행의 최댓값을 빼고 계산하면
   결과는 그대로이면서 안전해진다. 파이토치의 `nn.CrossEntropyLoss`도 내부에서 같은 처리를 한다.
2. **정답 클래스만 골라내기**: `probs[torch.arange(len(target)), target]`처럼 1장에서 배운
   **리스트를 인덱스로 사용하는 인덱싱**을 쓰면 샘플마다 정답 위치의 값을 한 번에 뽑을 수 있다.

직접 구현해 보면 `nn.CrossEntropyLoss`가 "소프트맥스를 포함하고 인덱스 정답을 받는다"는 본문 설명이
왜 그런지 코드 수준에서 이해된다.

### 문제 검토

- **적절성: 적합. 3장을 닫는 도전 문제로 잘 골랐다.** 본문에서 '이미 만들어져 있으니 가져다 쓰라'고 소개한 두 요소를
  직접 만들어 보게 해, 안에서 무슨 일이 벌어지는지 확인시킨다. 특히 교차 엔트로피를 구현해 보면
  "소프트맥스가 손실 함수 안에 있다", "정답은 인덱스로 받는다"는 본문의 두 주의사항이 자연스럽게 납득된다.
  1장의 `clamp()`와 인덱싱, 자동 미분까지 동원되므로 복습 효과도 크다.
- **[검토] 검증 방법을 지문에 넣으면 좋다.** 직접 구현한 것이 맞는지 확인할 방법이 없으면 도전 문제의 성취감이
  반감된다. 파이토치 제공 클래스와 결과를 비교하라는 한 구절이면 충분하다.
- **[검토] 오버플로 처리는 난도가 높다.** 최댓값을 빼는 처리를 모르면 큰 로짓에서 `nan`이 나올 수 있다.
  회오리 데이터 정도에서는 잘 드러나지 않지만, 힌트로 한 줄 남겨 두면 막히는 독자를 구할 수 있다.

**윤문안**

> **3-12**. [도전 문제] ReLU 활성화 함수와 교차 엔트로피 손실 함수를 직접 구현하고, 이를 사용해
> 회오리 모양 데이터 분류 모델을 만들어 보자. 그리고 같은 조건에서 파이토치가 제공하는 `nn.ReLU`,
> `nn.CrossEntropyLoss`로 학습한 결과와 비교해 보자.
>
> 힌트: 소프트맥스를 계산할 때 지수 함수의 값이 너무 커지지 않도록, 각 샘플의 로짓에서 최댓값을 빼고 시작하면 안전하다.